# Bond Data Cleaning Project
**Author:** Efe Izgialp  
**Description:** Cleaning and preparing corporate bond dataset for analysis using pandas

The Dirty Dataset
**Phase 1 | Python for Finance Roadmap**

---

## The Scenario

You've just been handed a CSV export of corporate bond issuance data — the kind of thing you might pull from Bloomberg, a data vendor, or a shared internal drive. It's 200+ rows of bond data covering issuers, coupons, ratings, maturities, and more.

There's one problem: it's a mess.

Before you can do any analysis on this data — compute average coupon by rating, look at sector exposure, calculate spread relationships — you need to make it usable. Your job in this assignment is to diagnose every problem in this dataset and fix it systematically.

**The deliverable:** A clean DataFrame, exported as `bond_issuance_clean.csv`, plus a notebook that documents every problem you found and every decision you made.

---

## Before You Write a Single Line of Code

Open the CSV in Excel or a text editor and look at it with your eyes first. What looks wrong? Make a list. Then come back and start coding.

This habit — look before you code — is one of the most important skills in data work.

---

## Your Tasks

Work through these in order. Each has a code cell below it. Use the markdown cells to explain what you found and what you decided to do.

1. **Load and inspect** — Load the CSV. What are the dimensions? What are the dtypes? Use `.info()`, `.describe()`, `.head()`, and `.sample()` to get oriented.
2. **Drop junk rows and columns** — Are there rows or columns that carry no data or contain metadata that snuck into the file?
3. **Handle duplicates** — Identify and remove exact duplicate rows.
4. **Standardize the `Issue_Date` column** — It's a mess of formats. Parse all of them into a consistent `datetime` dtype.
5. **Fix `Coupon_Rate`** — It should be a float. What's stopping that? Fix it.
6. **Fix `Face_Value_USD` and `Outstanding_Amt`** — Same problem, different flavour.
7. **Standardize `Issuer_Name`** — 'Apple Inc.', 'apple inc', 'APPLE INC.' are the same company. Fix it.
8. **Standardize `Sector` and `Currency`** — Similar problem.
9. **Handle `Credit_Rating`** — Multiple representations of 'no rating'. Pick one standard and apply it.
10. **Handle `Issue_Price`** — One value type is not a number. Decide what to do with it and document your reasoning.
11. **Handle missing values (`YTM` and others)** — Don't just drop them blindly. Think about which columns missing data matters for and why.
12. **Final check** — Run `.info()` and `.describe()` on your clean DataFrame. Do the dtypes look right? Do the value ranges make sense for bond data?
13. **Export** — Save the clean DataFrame to `bond_issuance_clean.csv`.

---

## A Note on Decisions

Data cleaning is not mechanical — it requires judgment. Every time you make a choice (e.g. "I'll drop rows where Issue_Date is missing" vs. "I'll try to infer the date from other columns"), write a sentence explaining *why*. That reasoning is what separates a junior analyst from a senior one.

---

## Step 1 — Load and Inspect

*Load the CSV and get oriented. What are you working with?*

In [200]:
import pandas as pd
import numpy as np

# Load the data

df = pd.read_csv('bond_issuance_dirty.csv')
df



,Issue_Date,Maturity_Date,Issuer_Name,Sector,Credit_Rating,Coupon_Rate,Face_Value_USD,Issue_Price,YTM,Currency,Outstanding_Amt,ISIN,Unnamed: 12,Notes
0,2011-04-18,2016-04-16,Ford Motor Co,Telecom,A-,2.129%,25000,95.1082,NaN,USD,85725000,US4311931853QG,NaN,NaN
1,2017-08-31,2022-08-30,ExxonMobil,Energy,A,3.536%,"500,000",89.3923,3.0088,Us Dollar,1882500000,US2351531223RK,NaN,NaN
2,2017-04-15,2027-04-13,Microsoft Corp,Energy,AA+,1.19,100000,87.3197,1.5269,EUR,370600000,US4040409964SE,NaN,NaN
3,20160112,2046-01-04,ExxonMobil,Telecom.,AA-,1.414%,500000,101.4657,1.0146,USD,24848000000,US4227004325HZ,NaN,NaN
4,03/07/2016,2021-03-06,APPLE INC.,Telecom,NaN,7.974,500000,90.3435,6.4817,USD,23841000000,US5933179746OI,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
213,2018-05-02,2038-04-27,Exxon Mobil Corp,Consumer Discretionary,NaN,7.17,"1,000,000",93.1623,8.5246,USD,2244000000,US2557724698ZJ,NaN,NaN
214,11/29/2013,2043-11-22,Ford Motor Company,Telecom.,B+,1.318,500000,104.9886,2.7175,USD,12312500000,US3069050782IH,NaN,NaN
215,21-Aug-2019,2049-08-13,ExxonMobil,Technology,AA,5.549,25000,104.9485,6.2893,EUR,713500000,US1124327519IB,NaN,NaN
216,2013-06-28,2023-06-26,Exxon Mobil Corp,FINANCIALS,B+,1.108,"10,000",87.038,1.9254,EUR,449430000,US2591589823NI,NaN,NaN


In [201]:
# .describe() and .info()

df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 218 entries, 0 to 217
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Issue_Date       213 non-null    object 
 1   Maturity_Date    213 non-null    object 
 2   Issuer_Name      213 non-null    object 
 3   Sector           212 non-null    object 
 4   Credit_Rating    184 non-null    object 
 5   Coupon_Rate      212 non-null    object 
 6   Face_Value_USD   212 non-null    object 
 7   Issue_Price      212 non-null    object 
 8   YTM              190 non-null    float64
 9   Currency         212 non-null    object 
 10  Outstanding_Amt  212 non-null    object 
 11  ISIN             212 non-null    object 
 12  Unnamed: 12      0 non-null      float64
 13  Notes            0 non-null      float64
dtypes: float64(3), object(11)
memory usage: 24.0+ KB


,YTM,Unnamed: 12,Notes
count,190.000000,0.0,0.0
mean,4.611454,NaN,NaN
std,2.556004,NaN,NaN
min,-0.443100,NaN,NaN
25%,2.422500,NaN,NaN
50%,4.880800,NaN,NaN
75%,6.842000,NaN,NaN
max,9.237300,NaN,NaN


In [202]:
df.sample()



,Issue_Date,Maturity_Date,Issuer_Name,Sector,Credit_Rating,Coupon_Rate,Face_Value_USD,Issue_Price,YTM,Currency,Outstanding_Amt,ISIN,Unnamed: 12,Notes
19,10/04/2018,2025-10-02,AT&T,technology,AA,5.189,"1,000",85.436,3.8824,USD,42435000,US2423995953BL,NaN,NaN


In [203]:
df.head()

,Issue_Date,Maturity_Date,Issuer_Name,Sector,Credit_Rating,Coupon_Rate,Face_Value_USD,Issue_Price,YTM,Currency,Outstanding_Amt,ISIN,Unnamed: 12,Notes
0,2011-04-18,2016-04-16,Ford Motor Co,Telecom,A-,2.129%,25000,95.1082,NaN,USD,85725000,US4311931853QG,NaN,NaN
1,2017-08-31,2022-08-30,ExxonMobil,Energy,A,3.536%,"500,000",89.3923,3.0088,Us Dollar,1882500000,US2351531223RK,NaN,NaN
2,2017-04-15,2027-04-13,Microsoft Corp,Energy,AA+,1.19,100000,87.3197,1.5269,EUR,370600000,US4040409964SE,NaN,NaN
3,20160112,2046-01-04,ExxonMobil,Telecom.,AA-,1.414%,500000,101.4657,1.0146,USD,24848000000,US4227004325HZ,NaN,NaN
4,03/07/2016,2021-03-06,APPLE INC.,Telecom,NaN,7.974,500000,90.3435,6.4817,USD,23841000000,US5933179746OI,NaN,NaN


**What did you find? Write your observations here before moving on.**

218 rows, 14 columns
almost everything is object (text). Dates, coupon rates, FV etc. probably because of %, signs, commas etc.
only TM is a number/float, everyhting else is unoperable to do calculations
YTM has a negative min? Check if that makes sense
Rating Agencies are unclear whether it's Moody's, S&P, Fitch.


---
## Step 2 — Drop Junk Rows and Columns

*Are there rows or columns that shouldn't be there at all?*

In [204]:
# Your code here

df_clean = df.dropna(how='all')
df_clean = df_clean[df_clean['Issue_Date'] != "Source: Bloomberg LP"]

df_clean


,Issue_Date,Maturity_Date,Issuer_Name,Sector,Credit_Rating,Coupon_Rate,Face_Value_USD,Issue_Price,YTM,Currency,Outstanding_Amt,ISIN,Unnamed: 12,Notes
0,2011-04-18,2016-04-16,Ford Motor Co,Telecom,A-,2.129%,25000,95.1082,NaN,USD,85725000,US4311931853QG,NaN,NaN
1,2017-08-31,2022-08-30,ExxonMobil,Energy,A,3.536%,"500,000",89.3923,3.0088,Us Dollar,1882500000,US2351531223RK,NaN,NaN
2,2017-04-15,2027-04-13,Microsoft Corp,Energy,AA+,1.19,100000,87.3197,1.5269,EUR,370600000,US4040409964SE,NaN,NaN
3,20160112,2046-01-04,ExxonMobil,Telecom.,AA-,1.414%,500000,101.4657,1.0146,USD,24848000000,US4227004325HZ,NaN,NaN
4,03/07/2016,2021-03-06,APPLE INC.,Telecom,NaN,7.974,500000,90.3435,6.4817,USD,23841000000,US5933179746OI,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
213,2018-05-02,2038-04-27,Exxon Mobil Corp,Consumer Discretionary,NaN,7.17,"1,000,000",93.1623,8.5246,USD,2244000000,US2557724698ZJ,NaN,NaN
214,11/29/2013,2043-11-22,Ford Motor Company,Telecom.,B+,1.318,500000,104.9886,2.7175,USD,12312500000,US3069050782IH,NaN,NaN
215,21-Aug-2019,2049-08-13,ExxonMobil,Technology,AA,5.549,25000,104.9485,6.2893,EUR,713500000,US1124327519IB,NaN,NaN
216,2013-06-28,2023-06-26,Exxon Mobil Corp,FINANCIALS,B+,1.108,"10,000",87.038,1.9254,EUR,449430000,US2591589823NI,NaN,NaN


In [205]:
df_clean['Issue_Date'].unique()


array(['2011-04-18', '2017-08-31', '2017-04-15', '20160112', '03/07/2016',
       '20130817', '2017-02-06', '2012-03-30', '04/20/2015',
       '29-Feb-2012', '30-Jan-2016', '2015-08-30', '2016-03-11',
       '2014-06-14', '2018-11-01', '17-Jan-2011', '20180425',
       '2015-09-10', '20191115', '10/04/2018', '05/10/2014', '12/07/2014',
       '20130105', '28-Jun-2015', '02-Feb-2010', '08/31/2011',
       '10/07/2020', '2019-05-12', '2019-12-15', '15-Dec-2019',
       '2018-01-26', '08/08/2010', '2016-12-26', '20120630', '20140413',
       '2020-08-07', '20111010', '2013-04-03', '20140422', '20101010',
       '2010-01-23', '09/30/2018', '20100907', '20180415', '2013-03-17',
       '20170713', '02/04/2017', '2018-05-02', '20111120', '02/14/2011',
       '20120224', '07/24/2015', '2016-09-06', '04-May-2017',
       '04-Mar-2017', '2017-11-23', '04/30/2010', '2018-12-12',
       '2018-06-01', '07/31/2012', '04/28/2012', '2016-12-12',
       '10-Dec-2014', '2011-04-17', '04/19/2015', '2015-

## **What did you drop and why?**

The rows that have missing data at every column is removed (fact checked via reduced number of rows)
There was also one metadata row, so we filtered that out by keeping all the rows without the word "Bloomberg"

---
## Step 3 — Handle Duplicates

In [206]:
# Your code here

df_clean = df_clean[~df.duplicated(subset=["Issue_Date", "Maturity_Date", "Issuer_Name", "Coupon_Rate"])]

df_clean


/var/folders/1g/r1xxhdtd0l15nzx4pz307_sr0000gn/T/ipykernel_59036/4085723513.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_clean = df_clean[~df.duplicated(subset=["Issue_Date", "Maturity_Date", "Issuer_Name", "Coupon_Rate"])]


,Issue_Date,Maturity_Date,Issuer_Name,Sector,Credit_Rating,Coupon_Rate,Face_Value_USD,Issue_Price,YTM,Currency,Outstanding_Amt,ISIN,Unnamed: 12,Notes
0,2011-04-18,2016-04-16,Ford Motor Co,Telecom,A-,2.129%,25000,95.1082,NaN,USD,85725000,US4311931853QG,NaN,NaN
1,2017-08-31,2022-08-30,ExxonMobil,Energy,A,3.536%,"500,000",89.3923,3.0088,Us Dollar,1882500000,US2351531223RK,NaN,NaN
2,2017-04-15,2027-04-13,Microsoft Corp,Energy,AA+,1.19,100000,87.3197,1.5269,EUR,370600000,US4040409964SE,NaN,NaN
3,20160112,2046-01-04,ExxonMobil,Telecom.,AA-,1.414%,500000,101.4657,1.0146,USD,24848000000,US4227004325HZ,NaN,NaN
4,03/07/2016,2021-03-06,APPLE INC.,Telecom,NaN,7.974,500000,90.3435,6.4817,USD,23841000000,US5933179746OI,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,09-Apr-2020,2023-04-09,Exxon Mobil Corp,technology,AA-,5.887,500,88.5869,6.8000,USD,1554500,US8923555411IO,NaN,NaN
214,11/29/2013,2043-11-22,Ford Motor Company,Telecom.,B+,1.318,500000,104.9886,2.7175,USD,12312500000,US3069050782IH,NaN,NaN
215,21-Aug-2019,2049-08-13,ExxonMobil,Technology,AA,5.549,25000,104.9485,6.2893,EUR,713500000,US1124327519IB,NaN,NaN
216,2013-06-28,2023-06-26,Exxon Mobil Corp,FINANCIALS,B+,1.108,"10,000",87.038,1.9254,EUR,449430000,US2591589823NI,NaN,NaN


**How many duplicates did you find? How did you define 'duplicate'?**

There were 10 duplicates that I got rid of. First I tried getting rid of duplicates based on ISIN number, which was the wrong approach because every ISIN is supposed to be different. Instead, I used multiple columns that define a bond to ensure I only got rid of duplicates. So, I filtered for issue date, maturity date, issuer name and coupon rate.


---
## Step 4 — Standardize Issue_Date

In [207]:
# First, look at the unique date formats present

df_clean

,Issue_Date,Maturity_Date,Issuer_Name,Sector,Credit_Rating,Coupon_Rate,Face_Value_USD,Issue_Price,YTM,Currency,Outstanding_Amt,ISIN,Unnamed: 12,Notes
0,2011-04-18,2016-04-16,Ford Motor Co,Telecom,A-,2.129%,25000,95.1082,NaN,USD,85725000,US4311931853QG,NaN,NaN
1,2017-08-31,2022-08-30,ExxonMobil,Energy,A,3.536%,"500,000",89.3923,3.0088,Us Dollar,1882500000,US2351531223RK,NaN,NaN
2,2017-04-15,2027-04-13,Microsoft Corp,Energy,AA+,1.19,100000,87.3197,1.5269,EUR,370600000,US4040409964SE,NaN,NaN
3,20160112,2046-01-04,ExxonMobil,Telecom.,AA-,1.414%,500000,101.4657,1.0146,USD,24848000000,US4227004325HZ,NaN,NaN
4,03/07/2016,2021-03-06,APPLE INC.,Telecom,NaN,7.974,500000,90.3435,6.4817,USD,23841000000,US5933179746OI,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,09-Apr-2020,2023-04-09,Exxon Mobil Corp,technology,AA-,5.887,500,88.5869,6.8000,USD,1554500,US8923555411IO,NaN,NaN
214,11/29/2013,2043-11-22,Ford Motor Company,Telecom.,B+,1.318,500000,104.9886,2.7175,USD,12312500000,US3069050782IH,NaN,NaN
215,21-Aug-2019,2049-08-13,ExxonMobil,Technology,AA,5.549,25000,104.9485,6.2893,EUR,713500000,US1124327519IB,NaN,NaN
216,2013-06-28,2023-06-26,Exxon Mobil Corp,FINANCIALS,B+,1.108,"10,000",87.038,1.9254,EUR,449430000,US2591589823NI,NaN,NaN


In [208]:
# Parse all formats into datetime
df_clean['Issue_Date'] = pd.to_datetime(df_clean['Issue_Date'], format ='mixed')

df_clean



/var/folders/1g/r1xxhdtd0l15nzx4pz307_sr0000gn/T/ipykernel_59036/2177347961.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Issue_Date'] = pd.to_datetime(df_clean['Issue_Date'], format ='mixed')


,Issue_Date,Maturity_Date,Issuer_Name,Sector,Credit_Rating,Coupon_Rate,Face_Value_USD,Issue_Price,YTM,Currency,Outstanding_Amt,ISIN,Unnamed: 12,Notes
0,2011-04-18,2016-04-16,Ford Motor Co,Telecom,A-,2.129%,25000,95.1082,NaN,USD,85725000,US4311931853QG,NaN,NaN
1,2017-08-31,2022-08-30,ExxonMobil,Energy,A,3.536%,"500,000",89.3923,3.0088,Us Dollar,1882500000,US2351531223RK,NaN,NaN
2,2017-04-15,2027-04-13,Microsoft Corp,Energy,AA+,1.19,100000,87.3197,1.5269,EUR,370600000,US4040409964SE,NaN,NaN
3,2016-01-12,2046-01-04,ExxonMobil,Telecom.,AA-,1.414%,500000,101.4657,1.0146,USD,24848000000,US4227004325HZ,NaN,NaN
4,2016-03-07,2021-03-06,APPLE INC.,Telecom,NaN,7.974,500000,90.3435,6.4817,USD,23841000000,US5933179746OI,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,2020-04-09,2023-04-09,Exxon Mobil Corp,technology,AA-,5.887,500,88.5869,6.8000,USD,1554500,US8923555411IO,NaN,NaN
214,2013-11-29,2043-11-22,Ford Motor Company,Telecom.,B+,1.318,500000,104.9886,2.7175,USD,12312500000,US3069050782IH,NaN,NaN
215,2019-08-21,2049-08-13,ExxonMobil,Technology,AA,5.549,25000,104.9485,6.2893,EUR,713500000,US1124327519IB,NaN,NaN
216,2013-06-28,2023-06-26,Exxon Mobil Corp,FINANCIALS,B+,1.108,"10,000",87.038,1.9254,EUR,449430000,US2591589823NI,NaN,NaN


**What formats did you encounter? How did you handle them?**

Lots of different formats with/without seperators, some have hyphens, some have slash, some are just numbers, some have letters for the month, DD/MM/YY format is mixed, years are 2 or 4 numbers.

I used pd.to_datetime() function specifically for the "Issue_Date" and only referenced that column within the dataframe to standardize it. I used "format='mixed'" given multiple different formats.

Note: one limitation of using above code is that I or pyhton does not know if a date 01/02/2026 is February 1st, or January 2nd. I need to go back to the source of confirmation (e.g. Bloomberg) to confirm the date convention used.


---
## Step 5 — Fix Coupon_Rate

In [209]:
# Inspect what's in this column
df_clean['Coupon_Rate'].unique()

#df_clean['Coupon_Rate'].dtype

array(['2.129%', '3.536%', '1.19', '1.414%', '7.974', '2.001', '8.211',
       '5.843', '0.652', '6.393', '7.855', '6.514%', '5.79', '4.069',
       '5.681', '3.137', '0.788%', '8.348', '1.878', '5.189', '7.812',
       '1.186', '6.498%', '3.316', '5.234', '2.581', '1.891', '6.464',
       '6.64', '5.765', '5.372', '5.785', '5.1%', '5.78', '3.774',
       '6.157', '7.392%', '6.269', '2.485%', '6.627', '0.902', '0.66',
       '5.546', '1.707', '7.616', '7.997', '4.84', '7.17', '1.882',
       '3.456', '2.498%', '8.091', '1.398%', '2.584', '1.196', '7.85%',
       '6.138', '2.521', '1.955', '8.232%', '7.367%', '7.775', '7.035',
       '1.222', '4.32', '7.25', '0.93', '7.656', '2.503', '2.696%',
       '4.064', '2.893%', '2.914', '4.204%', '0.919', '6.579', '2.267',
       '2.142', '3.226%', '1.246', '8.184', '1.571%', '3.211', '7.827',
       '6.475%', '4.981', '7.319%', '4.385', '7.524', '0.574', '3.29',
       '1.901', '2.231', '0.668', '0.898', '5.471', '8.269', '7.459',
       '6.013

In [210]:
# Clean and convert to float

df_clean['Coupon_Rate'] = df_clean['Coupon_Rate'].str.replace('%', '', regex=False).astype(float)





/var/folders/1g/r1xxhdtd0l15nzx4pz307_sr0000gn/T/ipykernel_59036/142395789.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Coupon_Rate'] = df_clean['Coupon_Rate'].str.replace('%', '', regex=False).astype(float)


In [211]:
df_clean['Coupon_Rate'].unique()

array([2.129, 3.536, 1.19 , 1.414, 7.974, 2.001, 8.211, 5.843, 0.652,
       6.393, 7.855, 6.514, 5.79 , 4.069, 5.681, 3.137, 0.788, 8.348,
       1.878, 5.189, 7.812, 1.186, 6.498, 3.316, 5.234, 2.581, 1.891,
       6.464, 6.64 , 5.765, 5.372, 5.785, 5.1  , 5.78 , 3.774, 6.157,
       7.392, 6.269, 2.485, 6.627, 0.902, 0.66 , 5.546, 1.707, 7.616,
       7.997, 4.84 , 7.17 , 1.882, 3.456, 2.498, 8.091, 1.398, 2.584,
       1.196, 7.85 , 6.138, 2.521, 1.955, 8.232, 7.367, 7.775, 7.035,
       1.222, 4.32 , 7.25 , 0.93 , 7.656, 2.503, 2.696, 4.064, 2.893,
       2.914, 4.204, 0.919, 6.579, 2.267, 2.142, 3.226, 1.246, 8.184,
       1.571, 3.211, 7.827, 6.475, 4.981, 7.319, 4.385, 7.524, 0.574,
       3.29 , 1.901, 2.231, 0.668, 0.898, 5.471, 8.269, 7.459, 6.013,
       3.1  , 3.948, 0.769, 7.075, 0.87 , 3.97 , 4.544, 8.345, 2.863,
       2.669, 2.795, 6.636, 6.635, 5.7  , 1.694, 3.984, 4.006, 4.878,
       5.917, 4.559, 7.912, 1.129, 4.694, 2.098, 7.603, 5.366, 6.53 ,
       7.697, 6.376,

**What was the problem and how did you fix it?**

First I looked at my column with the unique function to see the values, and observed that I need to remove "%" sign. In addition, I realized that the data type was an object whereas I need dtype float to be able to make calculations. So I replaced % with nothing, and converted the object type to float type


---
## Step 6 — Fix Face_Value_USD and Outstanding_Amt

In [212]:
print(df_clean['Face_Value_USD'].unique(), df_clean['Face_Value_USD'].dtype)
print(df_clean['Outstanding_Amt'].unique(), df_clean['Outstanding_Amt'].dtype)

['25000' '500,000' '100000' '500000' '50,000' '500' '1000' '10000'
 '250,000' '50000' '1,000' '250000' '5000' '1000000' '1,000,000' '10,000'
 '100,000' '5,000' '25,000'] object
['85725000' '1882500000' '370600000' '24848000000' '23841000000'
 '1453450000' '14232000' '19757000' '477600000' '290025000'
 '$2,675,000,000' '1029350000' '167975000' '2704000000' '274325000'
 '$49,270,000' '422600000' '113175000' '4555900000' '42435000'
 '$18,650,500' '2483750000' '1911950000' '11591750000' '12025500000'
 '$148,675,000' '45278000' '254050000' '43998000000' '68380000'
 '174285000' '$939,275,000' '116110000' '1839200000' '9486000000'
 '2030500' '1311500000' '338330000' '45351000000' '2949500000' '88380000'
 '5725000' '23425500000' '11303500' '3085750000' '15998000' '1847450000'
 '2244000000' '293290000' '99625000' '36166000' '1084800000' '17635000000'
 '90425000' '$1,534,700,000' '219900000' '42000000' '24735000' '184270000'
 '5795000000' '$57,675,000' '76775000' '42120000000' '2990400000'
 '$18

In [213]:
# Your code here

df_clean['Outstanding_Amt'] = (
    df_clean['Outstanding_Amt']
    .str.replace(r'[^\d]', '', regex=True)
    .astype(float))

df_clean['Face_Value_USD'] = (
    df_clean['Face_Value_USD']
    .str.replace(r'[^\d]', '', regex=True)
    .astype(float))






/var/folders/1g/r1xxhdtd0l15nzx4pz307_sr0000gn/T/ipykernel_59036/4119773526.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Outstanding_Amt'] = (
/var/folders/1g/r1xxhdtd0l15nzx4pz307_sr0000gn/T/ipykernel_59036/4119773526.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Face_Value_USD'] = (


In [214]:
print(df_clean['Face_Value_USD'].unique(), df_clean['Face_Value_USD'].dtype)
print(df_clean['Outstanding_Amt'].unique(), df_clean['Outstanding_Amt'].dtype)

[2.5e+04 5.0e+05 1.0e+05 5.0e+04 5.0e+02 1.0e+03 1.0e+04 2.5e+05 5.0e+03
 1.0e+06] float64
[8.572500e+07 1.882500e+09 3.706000e+08 2.484800e+10 2.384100e+10
 1.453450e+09 1.423200e+07 1.975700e+07 4.776000e+08 2.900250e+08
 2.675000e+09 1.029350e+09 1.679750e+08 2.704000e+09 2.743250e+08
 4.927000e+07 4.226000e+08 1.131750e+08 4.555900e+09 4.243500e+07
 1.865050e+07 2.483750e+09 1.911950e+09 1.159175e+10 1.202550e+10
 1.486750e+08 4.527800e+07 2.540500e+08 4.399800e+10 6.838000e+07
 1.742850e+08 9.392750e+08 1.161100e+08 1.839200e+09 9.486000e+09
 2.030500e+06 1.311500e+09 3.383300e+08 4.535100e+10 2.949500e+09
 8.838000e+07 5.725000e+06 2.342550e+10 1.130350e+07 3.085750e+09
 1.599800e+07 1.847450e+09 2.244000e+09 2.932900e+08 9.962500e+07
 3.616600e+07 1.084800e+09 1.763500e+10 9.042500e+07 1.534700e+09
 2.199000e+08 4.200000e+07 2.473500e+07 1.842700e+08 5.795000e+09
 5.767500e+07 7.677500e+07 4.212000e+10 2.990400e+09 1.880400e+08
 1.085525e+09 7.284000e+08 2.452000e+10 1.746500e+0

**What was the problem and how did you fix it?**

The problem was that some rows for boths columns have signs like , or %. Next, I removed all signs to keep floats so I can do some equations on the data. I used regex to define patterns, and learned that I can use "[^\d]" to remove all non numeric data.

---
## Step 7 — Standardize Issuer_Name

In [215]:
# First: look at the unique values

df_clean["Issuer_Name"].unique()


array(['Ford Motor Co', 'ExxonMobil', 'Microsoft Corp', 'APPLE INC.',
       'Microsoft Corporation', 'JPMorgan Chase & Co.', 'apple inc',
       'Exxon Mobil Corp', 'Goldman Sachs Group', 'JPMorgan Chase',
       'AT&T', 'Verizon Communications', 'Amazon', 'Ford Motor Company',
       'JP Morgan Chase', 'Goldman Sachs', 'FORD MOTOR CO', 'AT&T Inc',
       'Amazon.com Inc', 'Verizon', 'Apple Inc.'], dtype=object)

In [216]:
# Standardize

mapping = {
    'APPLE INC.' : 'Apple Inc.',  
    'apple inc' : 'Apple Inc.', 
    'Apple Inc.' : 'Apple Inc.',
    'Ford Motor Co' : 'Ford Motor Co.',  
    'FORD MOTOR CO' : 'Ford Motor Co.', 
    'Ford Motor Company' : 'Ford Motor Co.',
    'ExxonMobil' : 'Exxon Mobil Co.',
    'Exxon Mobil Corp':'Exxon Mobil Co.',
    'Microsoft Corp': 'Microsoft Co.',
    'Microsoft Corporation': 'Microsoft Co.',
    'JPMorgan Chase & Co.' : 'JPMorgan Chase',
    'JP Morgan Chase' : 'JPMorgan Chase',
    'Goldman Sachs Group' : 'Goldman Sachs',
    'AT&T' : 'AT&T Inc.',
    'AT&T Inc' : 'AT&T Inc.',
    'Verizon Communications' : 'Verizon',
    'Amazon.com': 'Amazon',
    'Amazon.com Inc': 'Amazon',
}

df_clean['Issuer_Name'] = df_clean['Issuer_Name'].replace(mapping)




/var/folders/1g/r1xxhdtd0l15nzx4pz307_sr0000gn/T/ipykernel_59036/1329729419.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Issuer_Name'] = df_clean['Issuer_Name'].replace(mapping)


In [217]:
df_clean["Issuer_Name"].unique()

array(['Ford Motor Co.', 'Exxon Mobil Co.', 'Microsoft Co.', 'Apple Inc.',
       'JPMorgan Chase', 'Goldman Sachs', 'AT&T Inc.', 'Verizon',
       'Amazon'], dtype=object)

**What approach did you take? Did you use a mapping, fuzzy matching, or something else? Why?**

I used a mapping dictionary because dataset has a 9 different issuers. It is more direct and deterministic than fuzzy method and that would be better suited to larger datasets.


In [218]:
df_clean

,Issue_Date,Maturity_Date,Issuer_Name,Sector,Credit_Rating,Coupon_Rate,Face_Value_USD,Issue_Price,YTM,Currency,Outstanding_Amt,ISIN,Unnamed: 12,Notes
0,2011-04-18,2016-04-16,Ford Motor Co.,Telecom,A-,2.129,25000.0,95.1082,NaN,USD,8.572500e+07,US4311931853QG,NaN,NaN
1,2017-08-31,2022-08-30,Exxon Mobil Co.,Energy,A,3.536,500000.0,89.3923,3.0088,Us Dollar,1.882500e+09,US2351531223RK,NaN,NaN
2,2017-04-15,2027-04-13,Microsoft Co.,Energy,AA+,1.190,100000.0,87.3197,1.5269,EUR,3.706000e+08,US4040409964SE,NaN,NaN
3,2016-01-12,2046-01-04,Exxon Mobil Co.,Telecom.,AA-,1.414,500000.0,101.4657,1.0146,USD,2.484800e+10,US4227004325HZ,NaN,NaN
4,2016-03-07,2021-03-06,Apple Inc.,Telecom,NaN,7.974,500000.0,90.3435,6.4817,USD,2.384100e+10,US5933179746OI,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,2020-04-09,2023-04-09,Exxon Mobil Co.,technology,AA-,5.887,500.0,88.5869,6.8000,USD,1.554500e+06,US8923555411IO,NaN,NaN
214,2013-11-29,2043-11-22,Ford Motor Co.,Telecom.,B+,1.318,500000.0,104.9886,2.7175,USD,1.231250e+10,US3069050782IH,NaN,NaN
215,2019-08-21,2049-08-13,Exxon Mobil Co.,Technology,AA,5.549,25000.0,104.9485,6.2893,EUR,7.135000e+08,US1124327519IB,NaN,NaN
216,2013-06-28,2023-06-26,Exxon Mobil Co.,FINANCIALS,B+,1.108,10000.0,87.038,1.9254,EUR,4.494300e+08,US2591589823NI,NaN,NaN


---
## Step 8 — Standardize Sector and Currency

In [219]:
print(df_clean)
print(df_clean["Sector"].unique())
print(df_clean["Currency"].unique())

    Issue_Date Maturity_Date      Issuer_Name      Sector Credit_Rating  \
0   2011-04-18    2016-04-16   Ford Motor Co.     Telecom            A-   
1   2017-08-31    2022-08-30  Exxon Mobil Co.      Energy             A   
2   2017-04-15    2027-04-13    Microsoft Co.      Energy           AA+   
3   2016-01-12    2046-01-04  Exxon Mobil Co.    Telecom.           AA-   
4   2016-03-07    2021-03-06       Apple Inc.     Telecom           NaN   
..         ...           ...              ...         ...           ...   
212 2020-04-09    2023-04-09  Exxon Mobil Co.  technology           AA-   
214 2013-11-29    2043-11-22   Ford Motor Co.    Telecom.            B+   
215 2019-08-21    2049-08-13  Exxon Mobil Co.  Technology            AA   
216 2013-06-28    2023-06-26  Exxon Mobil Co.  FINANCIALS            B+   
217 2013-04-24    2023-04-22       Apple Inc.     Energy            AA+   

     Coupon_Rate  Face_Value_USD Issue_Price     YTM   Currency  \
0          2.129         25000.0

In [220]:
# Your code here

#Standardizing Currency
mapping_currency = {
    'Us Dollar' : 'USD',
    'usd' : 'USD'
}

df_clean['Currency'] = df_clean['Currency'].replace(mapping_currency)

#Standardizing Sector

mapping_sector = {
    'Telecom.' : 'Telecom',
    'financials' : 'Financials',
    'FINANCIALS' : 'Financials',
    'Energy ' : 'Energy',
    'technology' : 'Technology'
}

df_clean['Sector'] = df_clean['Sector'].replace(mapping_sector)




/var/folders/1g/r1xxhdtd0l15nzx4pz307_sr0000gn/T/ipykernel_59036/3687489530.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Currency'] = df_clean['Currency'].replace(mapping_currency)
/var/folders/1g/r1xxhdtd0l15nzx4pz307_sr0000gn/T/ipykernel_59036/3687489530.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Sector'] = df_clean['Sector'].replace(mapping_sector)


In [221]:
print(df_clean["Sector"].unique())
print(df_clean["Currency"].unique())

['Telecom' 'Energy' 'Financials' 'Technology' 'Consumer Discretionary']
['USD' 'EUR' 'GBP']


**What did you find and how did you fix it?**

Fixed formatting issues for capitalization, trailing spaces and punctuation for both sector and currency using dictionary mapping.

Note: There are wrong sector assignments. E.g. ExxonMobil listed as Telecom or Apple as Financials. This needs be automated, but not as part of this assignment.


---
## Step 9 — Handle Credit_Rating

In [222]:
# What unique values are present?

print(df_clean['Credit_Rating'].unique())
print(df_clean['Credit_Rating'].isna().sum())
print((df_clean['Credit_Rating'] == 'NR').sum())




['A-' 'A' 'AA+' 'AA-' nan 'aaa' 'AA' 'AAA' 'NR' 'Aaa' 'A+' 'BB+' 'B+'
 'BBB-' 'BBB+' 'BBB' 'BB']
24
12


In [223]:
# Standardize

df_clean['Credit_Rating'] = df_clean['Credit_Rating'].replace({
    'aaa': 'AAA',
    'Aaa': 'AAA',
})

df_clean["Credit_Rating"] = df_clean['Credit_Rating'].fillna('NR')

df_clean


/var/folders/1g/r1xxhdtd0l15nzx4pz307_sr0000gn/T/ipykernel_59036/1464055061.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Credit_Rating'] = df_clean['Credit_Rating'].replace({
/var/folders/1g/r1xxhdtd0l15nzx4pz307_sr0000gn/T/ipykernel_59036/1464055061.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean["Credit_Rating"] = df_clean['Credit_Rating'].fillna('NR')


,Issue_Date,Maturity_Date,Issuer_Name,Sector,Credit_Rating,Coupon_Rate,Face_Value_USD,Issue_Price,YTM,Currency,Outstanding_Amt,ISIN,Unnamed: 12,Notes
0,2011-04-18,2016-04-16,Ford Motor Co.,Telecom,A-,2.129,25000.0,95.1082,NaN,USD,8.572500e+07,US4311931853QG,NaN,NaN
1,2017-08-31,2022-08-30,Exxon Mobil Co.,Energy,A,3.536,500000.0,89.3923,3.0088,USD,1.882500e+09,US2351531223RK,NaN,NaN
2,2017-04-15,2027-04-13,Microsoft Co.,Energy,AA+,1.190,100000.0,87.3197,1.5269,EUR,3.706000e+08,US4040409964SE,NaN,NaN
3,2016-01-12,2046-01-04,Exxon Mobil Co.,Telecom,AA-,1.414,500000.0,101.4657,1.0146,USD,2.484800e+10,US4227004325HZ,NaN,NaN
4,2016-03-07,2021-03-06,Apple Inc.,Telecom,NR,7.974,500000.0,90.3435,6.4817,USD,2.384100e+10,US5933179746OI,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,2020-04-09,2023-04-09,Exxon Mobil Co.,Technology,AA-,5.887,500.0,88.5869,6.8000,USD,1.554500e+06,US8923555411IO,NaN,NaN
214,2013-11-29,2043-11-22,Ford Motor Co.,Telecom,B+,1.318,500000.0,104.9886,2.7175,USD,1.231250e+10,US3069050782IH,NaN,NaN
215,2019-08-21,2049-08-13,Exxon Mobil Co.,Technology,AA,5.549,25000.0,104.9485,6.2893,EUR,7.135000e+08,US1124327519IB,NaN,NaN
216,2013-06-28,2023-06-26,Exxon Mobil Co.,Financials,B+,1.108,10000.0,87.038,1.9254,EUR,4.494300e+08,US2591589823NI,NaN,NaN


**What representations of 'no rating' did you find? What did you standardize to and why?**

First, I fixed the inconsistent labels for the same rating (e.g. aaa vs. Aaa vs. AAA) and standardized them using mapping. I also identified two representation of missing ratings, NR and blank values. NR likely means not rated which is typical in bond data. I treated missing values as unrated rather than removing them given that these rows accounted for 18% of the data set.

---
## Step 10 — Handle Issue_Price

In [224]:
# Inspect the column
print(df_clean["Issue_Price"])
print(df_clean["Issue_Price"].unique())


0       95.1082
1       89.3923
2       87.3197
3      101.4657
4       90.3435
         ...   
212     88.5869
214    104.9886
215    104.9485
216      87.038
217    100.5148
Name: Issue_Price, Length: 200, dtype: object
['95.1082' '89.3923' '87.3197' '101.4657' '90.3435' '97.4527' '95.6073'
 '95.9999' '101.6993' '96.3556' '88.1487' '100.3163' '104.7044' '96.7376'
 '101.5326' '93.2162' '96.6502' '99.9463' '87.8771' '85.436' '99.0691'
 '101.1636' '94.0939' '97.3473' '104.9557' '85.6216' '88.1307' '86.0739'
 '99.4755' '103.5753' '92.4783' '89.751' '90.8254' '91.5652' '102.8569'
 '89.3744' '99.684' '100.2624' '97.0002' '92.4892' '103.3494' '94.4819'
 '100.8969' '94.0105' '102.8927' '99.9376' '99.2071' '93.1623' '103.1661'
 '93.7637' '93.8626' '101.8323' '103.0578' '94.2752' '85.6357' '102.341'
 '90.5669' '99.3047' '92.0965' '94.2166' '104.7505' '97.9848' '87.768'
 '85.2784' '100.1182' '100.6975' '89.0617' '90.3662' '98.2902' '97.2886'
 '90.9287' '98.8063' '104.7139' '93.4826' '86.3182' '

In [225]:
# Fix it

df_clean['Issue_Price'] = df_clean['Issue_Price'].replace('Par', '100')
df_clean['Issue_Price'] = df_clean['Issue_Price'].astype(float)

/var/folders/1g/r1xxhdtd0l15nzx4pz307_sr0000gn/T/ipykernel_59036/815370804.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Issue_Price'] = df_clean['Issue_Price'].replace('Par', '100')
/var/folders/1g/r1xxhdtd0l15nzx4pz307_sr0000gn/T/ipykernel_59036/815370804.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Issue_Price'] = df_clean['Issue_Price'].astype(float)


In [226]:
#(df_clean['Issue_Price'] == 100).sum()

**What non-numeric value did you find? What did you decide to do with it and why? (Hint: think about what 'Par' means in bond terms.)**

There was one row with a value of 'Par', so I converted 'Par' to '100' first, and then converted the column data type from object to float to be able to make calculations later. Par value means bond is at FV of 100, likely at issuance or in a very unlikely situation where Coupon = YTM.


---
## Step 11 — Handle Missing Values

In [194]:
# Count missing values per column

df_clean.isna().sum()


Issue_Date           0
Maturity_Date        0
Issuer_Name          0
Sector               0
Credit_Rating        0
Coupon_Rate          0
Face_Value_USD       0
Issue_Price          0
YTM                 22
Currency             0
Outstanding_Amt      0
ISIN                 0
Unnamed: 12        200
Notes              200
dtype: int64

In [227]:
# Make decisions and apply them


#Remove useless columns Unnamed: 12 and Notes
df_clean = df_clean.drop(columns=['Unnamed: 12', 'Notes'])

#Remove the 22 rows corresponding to blank YTM
df_clean.dropna(subset=['YTM'], inplace=True)
df_clean.isna().sum()







Issue_Date         0
Maturity_Date      0
Issuer_Name        0
Sector             0
Credit_Rating      0
Coupon_Rate        0
Face_Value_USD     0
Issue_Price        0
YTM                0
Currency           0
Outstanding_Amt    0
ISIN               0
dtype: int64

**Column by column: what was missing, what did you do, and why?**

'YTM' is a critical variable for analysis. Rows with blank values are useless, so I removed 22 those corresponding 22 rows.
Also "Unnamed: 12' and 'Notes' don't bring any value and removed accordingly. 

---
## Step 12 — Final Check

In [228]:
# Run .info() and .describe() on the clean DataFrame
# Do the dtypes look right?
# Do the value ranges make sense for bond data?

df_clean.info()



<class 'pandas.core.frame.DataFrame'>
Index: 178 entries, 1 to 217
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Issue_Date       178 non-null    datetime64[ns]
 1   Maturity_Date    178 non-null    object        
 2   Issuer_Name      178 non-null    object        
 3   Sector           178 non-null    object        
 4   Credit_Rating    178 non-null    object        
 5   Coupon_Rate      178 non-null    float64       
 6   Face_Value_USD   178 non-null    float64       
 7   Issue_Price      178 non-null    float64       
 8   YTM              178 non-null    float64       
 9   Currency         178 non-null    object        
 10  Outstanding_Amt  178 non-null    float64       
 11  ISIN             178 non-null    object        
dtypes: datetime64[ns](1), float64(5), object(6)
memory usage: 18.1+ KB


In [230]:
df_clean.describe()

,Issue_Date,Coupon_Rate,Face_Value_USD,Issue_Price,YTM,Outstanding_Amt
count,178,178.000000,178.000000,178.000000,178.000000,1.780000e+02
mean,2015-12-14 07:49:12.808988672,4.569781,210303.370787,95.481678,4.589112,5.792054e+09
min,2010-01-23 00:00:00,0.574000,500.000000,85.278400,-0.443100,1.554500e+06
25%,2013-07-06 00:00:00,2.489500,5000.000000,90.599825,2.422500,9.502750e+07
50%,2016-01-04 12:00:00,4.582500,50000.000000,96.113400,4.721450,7.209500e+08
75%,2018-08-01 00:00:00,6.612250,250000.000000,100.339325,6.744650,4.486425e+09
max,2020-12-05 00:00:00,8.348000,1000000.000000,104.988600,9.237300,4.866000e+10
std,NaN,2.448938,330380.055228,5.814609,2.535379,1.082675e+10


**Does anything still look off? Any values that seem out of range for a bond dataset?**

Negative YTM is unusual but other than that all values (Coupons, FV, Price etc.) seems to be consistent with bond characteristics. 


---
## Step 13 — Export

In [233]:
# Save the clean DataFrame
# df_clean.to_csv('bond_issuance_clean.csv', index=False)

df_clean.to_csv('bond_clean.csv', index=False)


 #For Excel use the code below:
    #df_clean.to_excel('bond_clean.xlsx', index=False)


---
## Wrap-Up

When you're done, write a short summary below (3–5 sentences) as if you're sending an email to a colleague explaining what was wrong with the data and what you did to fix it. No bullet points — just write it out.


This was a typical data cleaning exercise where the data had multiple categorization, punctuation and blank value across different rows and columns for standard corporate bond characteristics across eight different issuers. I standardized categorical variables such as issuer names and credit ratings, removing observation based on importance (kept unrated bonds but got rid of those without YTM values). I also fixed numeric fields like issue price, or coupon percentages while getting rid of empty/irrelevant columns. Final dataset is clean and ready for analysis.